In [1]:
import pandas as pd
import altair as alt
from scipy.stats import pearsonr
import itertools
import math

In [2]:
# raise data limit for altair plot
alt.data_transformers.disable_max_rows()


DataTransformerRegistry.enable('default')

In [20]:
# summary_df = '../results/summaries/HA_binding.csv'

In [2]:
summary_df = None
corr_chart = None

In [21]:
# --- load data ---
summary = pd.read_csv(summary_df)

In [ ]:
# Create mutation column
summary['mutation'] = summary['wildtype'].astype(str) + summary['site'].astype(str) + summary['mutant'].astype(str)

# Select columns - the x/y pair to plot
columns = [
    'H5 HA binding escape',
    'entry in 293T_H5_HA cells'
]
col_a, col_b = columns

# Display titles for the axes (column name -> axis title)
axis_titles = {
    'H5 HA binding escape': 'H5 HA binding',
    'entry in 293T_H5_HA cells': 'entry in 293T_H5_HA cells',
}

# Keep only sites whose structure_site ends in _a or _b
summary = summary[
    summary['structure_site'].astype(str).str.endswith(('_a', '_b'))
].copy()

# Assign chain based on structure_site suffix
summary['chain'] = summary['structure_site'].astype(str).str[-2:].map(
    {'_a': 'alpha', '_b': 'beta'}
)

# Clean data
df_clean = summary.dropna(subset=columns + ['mutation', 'region', 'chain']).copy()

# Create tooltips with ALL columns
all_tooltips = [
    alt.Tooltip('mutation:N', title='Mutation'),
    alt.Tooltip('region:N', title='Region'),
    alt.Tooltip('H5 HA binding escape:Q', format=".3f"),
    alt.Tooltip('entry in 293T_H5_HA cells:Q', format=".3f")
]

# Build one scatter plot per chain
def make_scatter(chain_df, chain_name):
    # Calculate Pearson R for this chain
    valid_data = chain_df[[col_a, col_b]].dropna()
    r, p = pearsonr(valid_data[col_a], valid_data[col_b])

    # Per-chain interaction selections (unique names avoid clashes between charts)
    mut_selection = alt.selection_point(
        on="mouseover", fields=['mutation'], empty=False, name=f"mut_{chain_name}"
    )
    region_selection = alt.selection_point(
        fields=['region'], bind='legend', toggle='true', name=f"region_{chain_name}"
    )

    base_chart = (
        alt.Chart(chain_df)
        .add_params(mut_selection, region_selection)
        .transform_filter(
            f'isValid(datum["{col_a}"]) && isValid(datum["{col_b}"])'
        )
        .encode(
            x=alt.X(
                f"{col_a}:Q",
                title=axis_titles[col_a],
                scale=alt.Scale(padding=7, nice=False, zero=False),
            ),
            y=alt.Y(
                f"{col_b}:Q",
                title=axis_titles[col_b],
                scale=alt.Scale(padding=7, nice=False, zero=False),
            ),
        )
        .properties(
            width=250,
            height=250,
            title=f"{chain_name} chain",
        )
    )

    # Points with interactivity and color by region
    foreground = base_chart.mark_point(
        filled=True,
        fillOpacity=0.6,
        stroke="black",
        strokeOpacity=1,
    ).encode(
        color=alt.Color('region:N', legend=alt.Legend(title='Region')),
        tooltip=all_tooltips,
        strokeWidth=alt.condition(mut_selection, alt.value(2), alt.value(0)),
        size=alt.condition(mut_selection, alt.value(100), alt.value(50)),
        opacity=alt.condition(region_selection, alt.value(0.6), alt.value(0.1))
    )

    # Add R value text
    r_text = (
        alt.Chart(pd.DataFrame({'r_text': [f'r = {r:.2f}']}))
        .mark_text(size=14, align="left", baseline="top", color="black", fontWeight="bold")
        .encode(
            text='r_text:N',
            x=alt.value(5),
            y=alt.value(5),
        )
    )

    return foreground + r_text

scatters = [
    make_scatter(df_clean[df_clean['chain'] == chain_name], chain_name)
    for chain_name in ['alpha', 'beta']
]

# Arrange the two chain scatter plots side by side
chart = (
    alt.hconcat(*scatters)
    .configure_axis(
        grid=False, 
        titleFontSize=11, 
        labelFontSize=10, 
        labelOverlap="greedy", 
        titleFontWeight="normal"
    )
    .configure_legend(
        titleFontSize=12,
        labelFontSize=11
    )
)

chart.save(corr_chart)
chart